# 数仓建模 — 高级数据工程师面试精讲

本笔记覆盖系统设计 & 架构模块中最常考的数仓建模知识点。

| 主题 | 出现频率 |
|------|----------|
| Kimball：星型 / 雪花型 / Galaxy 模式 | 高频 |
| SCD Type 1/2/3/4/6 | 高频 |
| Data Vault 2.0：Hub / Link / Satellite | 重要 |
| One Big Table (OBT) vs 规范化 | 重要 |
| Lakehouse 架构设计 | 高频 |

---
## 1. Kimball 维度建模：星型 / 雪花型 / Galaxy

### 核心思想

Ralph Kimball 的维度建模方法论以**事实表（Fact Table）+ 维度表（Dimension Table）**为核心，
目标是让业务用户能快速理解并查询数据。

### 星型模式（Star Schema）

```
               dim_date
                  │
   dim_product ──fact_sales── dim_customer
                  │
               dim_store
```

- 事实表居中，维度表直接相连，**非规范化**（维度冗余）
- 查询时 JOIN 少（最多 1 跳），BI 工具友好
- 存储冗余，维度更新需更新所有冗余列

### 雪花型模式（Snowflake Schema）

```
   dim_category
        │
   dim_product ──fact_sales── dim_customer── dim_city── dim_country
        │
   dim_brand
```

- 维度表进一步**规范化**，拆分为多层层级表
- 存储节省，数据一致性好，但查询 JOIN 增加
- 适合维度层级深、ETL 复杂度可接受的场景

### Galaxy / Constellation 模式

```
   dim_date ──fact_sales────────── dim_product
       │                               │
   dim_date ──fact_inventory── dim_product  (共享维度)
```

- 多个事实表**共享**同一批维度表
- 适合企业级 DWH（多业务主题），需严格管理一致性维度（Conformed Dimensions）

### 对比总结

| 维度 | 星型 | 雪花型 | Galaxy |
|------|------|--------|--------|
| 规范化程度 | 低（冗余） | 高（规范化） | 混合 |
| 查询性能 | 好（少 JOIN） | 较差（多 JOIN） | 取决于设计 |
| 存储占用 | 较大 | 较小 | 较大 |
| 维护复杂度 | 低 | 中 | 高 |
| BI 工具友好度 | 极好 | 一般 | 好 |
| 适用场景 | 单主题快速分析 | 维度层级深 | 多主题企业级 |

In [ ]:
# Demonstrate Kimball schema design with DuckDB (in-process, no server needed)
import duckdb

con = duckdb.connect(":memory:")

# ── Star Schema DDL ──────────────────────────────────────────────────────────
con.execute("""
-- Dimension: Date (denormalized, no separate month/year tables)
CREATE TABLE dim_date (
    date_key       INTEGER PRIMARY KEY,  -- surrogate key: 20240101
    full_date      DATE NOT NULL,
    year           INTEGER,
    quarter        INTEGER,
    month          INTEGER,
    month_name     VARCHAR(10),
    week_of_year   INTEGER,
    day_of_week    INTEGER,
    is_weekend     BOOLEAN
);

-- Dimension: Product (denormalized — category & brand attributes embedded)
CREATE TABLE dim_product (
    product_key    INTEGER PRIMARY KEY,
    product_id     VARCHAR(20) NOT NULL,  -- natural/business key
    product_name   VARCHAR(200),
    category       VARCHAR(100),
    subcategory    VARCHAR(100),
    brand          VARCHAR(100),
    unit_cost      DECIMAL(10,2)
);

-- Dimension: Customer
CREATE TABLE dim_customer (
    customer_key   INTEGER PRIMARY KEY,
    customer_id    VARCHAR(20) NOT NULL,
    full_name      VARCHAR(200),
    email          VARCHAR(200),
    city           VARCHAR(100),
    country        VARCHAR(100),
    segment        VARCHAR(50)   -- e.g., 'RETAIL', 'WHOLESALE'
);

-- Fact table: Sales (all FKs + additive measures)
CREATE TABLE fact_sales (
    sale_id        BIGINT PRIMARY KEY,
    date_key       INTEGER REFERENCES dim_date(date_key),
    product_key    INTEGER REFERENCES dim_product(product_key),
    customer_key   INTEGER REFERENCES dim_customer(customer_key),
    quantity       INTEGER,
    unit_price     DECIMAL(10,2),
    gross_amount   DECIMAL(12,2),  -- pre-calculated measure
    discount_amt   DECIMAL(10,2),
    net_amount     DECIMAL(12,2)
);
""")

# Seed sample data
con.execute("""
INSERT INTO dim_date VALUES
    (20240101, '2024-01-01', 2024, 1, 1, 'January', 1, 1, FALSE),
    (20240102, '2024-01-02', 2024, 1, 1, 'January', 1, 2, FALSE),
    (20240103, '2024-01-03', 2024, 1, 1, 'January', 1, 3, FALSE);

INSERT INTO dim_product VALUES
    (1, 'P001', 'Laptop Pro 15', 'Electronics', 'Computers', 'TechBrand', 800.00),
    (2, 'P002', 'Wireless Mouse', 'Electronics', 'Accessories', 'TechBrand', 15.00),
    (3, 'P003', 'USB-C Cable',   'Electronics', 'Accessories', 'CableCo',   5.00);

INSERT INTO dim_customer VALUES
    (1, 'C001', 'Alice Wang',  'alice@example.com', 'Shanghai', 'China', 'RETAIL'),
    (2, 'C002', 'Bob Zhang',   'bob@example.com',   'Beijing',  'China', 'WHOLESALE'),
    (3, 'C003', 'Carol Li',    'carol@example.com', 'NYC',      'USA',   'RETAIL');

INSERT INTO fact_sales VALUES
    (1, 20240101, 1, 1, 2, 1200.00, 2400.00, 100.00, 2300.00),
    (2, 20240101, 2, 1, 5,   25.00,  125.00,   5.00,  120.00),
    (3, 20240102, 3, 2, 10,  10.00,  100.00,   0.00,  100.00),
    (4, 20240103, 1, 3, 1, 1200.00, 1200.00,  50.00, 1150.00);
""")

# Typical star schema query: sales by product category and month
result = con.execute("""
    SELECT
        p.category,
        d.month_name,
        SUM(f.net_amount)   AS total_net_sales,
        SUM(f.quantity)     AS total_units,
        COUNT(*)            AS num_transactions
    FROM fact_sales f
    JOIN dim_product  p ON f.product_key  = p.product_key
    JOIN dim_date     d ON f.date_key     = d.date_key
    JOIN dim_customer c ON f.customer_key = c.customer_key
    GROUP BY p.category, d.month_name
    ORDER BY total_net_sales DESC
""").fetchdf()

print("=== Star Schema: Sales by Category & Month ===")
print(result.to_string(index=False))

In [ ]:
# Snowflake Schema: normalize dim_product into category/brand sub-tables

con.execute("""
-- Sub-dimension: Category (normalized out of dim_product)
CREATE TABLE dim_category (
    category_key    INTEGER PRIMARY KEY,
    category_name   VARCHAR(100),
    subcategory     VARCHAR(100)
);

-- Sub-dimension: Brand
CREATE TABLE dim_brand (
    brand_key       INTEGER PRIMARY KEY,
    brand_name      VARCHAR(100),
    brand_country   VARCHAR(100)
);

-- Snowflake dim_product: references category & brand instead of embedding them
CREATE TABLE dim_product_snow (
    product_key     INTEGER PRIMARY KEY,
    product_id      VARCHAR(20) NOT NULL,
    product_name    VARCHAR(200),
    category_key    INTEGER REFERENCES dim_category(category_key),
    brand_key       INTEGER REFERENCES dim_brand(brand_key),
    unit_cost       DECIMAL(10,2)
);
""")

con.execute("""
INSERT INTO dim_category VALUES (1, 'Electronics', 'Computers'), (2, 'Electronics', 'Accessories');
INSERT INTO dim_brand    VALUES (1, 'TechBrand', 'USA'), (2, 'CableCo', 'China');
INSERT INTO dim_product_snow VALUES
    (1, 'P001', 'Laptop Pro 15',  1, 1, 800.00),
    (2, 'P002', 'Wireless Mouse', 2, 1,  15.00),
    (3, 'P003', 'USB-C Cable',    2, 2,   5.00);
""")

# Snowflake query requires extra JOIN to sub-dimensions
result_snow = con.execute("""
    SELECT
        c.category_name,
        b.brand_name,
        SUM(f.net_amount) AS total_net_sales
    FROM fact_sales f
    JOIN dim_product_snow p ON f.product_key  = p.product_key
    JOIN dim_category     c ON p.category_key = c.category_key
    JOIN dim_brand        b ON p.brand_key    = b.brand_key
    JOIN dim_date         d ON f.date_key     = d.date_key
    GROUP BY c.category_name, b.brand_name
    ORDER BY total_net_sales DESC
""").fetchdf()

print("=== Snowflake Schema: Sales by Category & Brand (extra JOINs) ===")
print(result_snow.to_string(index=False))
print("\nNote: Snowflake needs JOIN to dim_category AND dim_brand (vs star: just dim_product)")

---
## 2. SCD（缓慢变化维度）Types 1 / 2 / 3 / 4 / 6

**SCD（Slowly Changing Dimension）** 解决维度属性随时间变化时如何保存历史的问题。

以客户地址变更为例：

### Type 1 — 直接覆盖（无历史）
- 用新值覆盖旧值，**不保留历史**
- 适用：数据纠错、不需要历史追踪的属性（如拼写更正）

### Type 2 — 新增行 + 有效期（最常用）
- 插入新行，用 `effective_date` / `expiry_date` 标识有效期
- 当前记录用 `is_current = TRUE` 或 `expiry_date = '9999-12-31'`
- **完整历史可追溯**，dbt snapshot 使用此模式

### Type 3 — 增加列（有限历史）
- 在同一行增加 `previous_value` 列，只保存前一个值
- 适用：只需比较「当前 vs 上一次」，不需要完整历史

### Type 4 — 独立历史表
- 主表只保留当前值，历史表保存所有历史版本
- 查询当前值快，历史查询需 JOIN 历史表

### Type 6 — 混合（1 + 2 + 3）
- 结合 Type 2 的多行 + Type 3 的 `current_value` 列
- 每行既有有效期（Type 2），又有快捷访问当前值的列（Type 3），覆写旧行的 current_value（Type 1）
- **最灵活**，也最复杂

| 类型 | 历史 | 实现复杂度 | 存储 | 典型场景 |
|------|------|-----------|------|----------|
| Type 1 | 无 | 低 | 小 | 数据纠错 |
| Type 2 | 全量 | 中 | 大 | 地址/状态变更追踪 |
| Type 3 | 前一次 | 低 | 小 | 只需比较前后两次 |
| Type 4 | 全量（独立表） | 中 | 中 | 访问模式分离 |
| Type 6 | 全量 + 当前列 | 高 | 大 | 复杂分析需求 |

In [ ]:
import duckdb
import pandas as pd
from datetime import date

con2 = duckdb.connect(":memory:")

# ── SCD Type 1: Overwrite ────────────────────────────────────────────────────
con2.execute("""
CREATE TABLE scd1_customer (
    customer_id  VARCHAR(20) PRIMARY KEY,
    full_name    VARCHAR(200),
    city         VARCHAR(100),
    updated_at   TIMESTAMP DEFAULT NOW()
);
INSERT INTO scd1_customer VALUES ('C001', 'Alice Wang', 'Shanghai', '2024-01-01 00:00:00');
""")

print("=== SCD Type 1: Before update ===")
print(con2.execute("SELECT * FROM scd1_customer").fetchdf().to_string(index=False))

# Type 1: MERGE / UPDATE — just overwrite
con2.execute("""
INSERT OR REPLACE INTO scd1_customer VALUES ('C001', 'Alice Wang', 'Beijing', NOW());
""")
print("\n=== SCD Type 1: After update (Shanghai → Beijing, history LOST) ===")
print(con2.execute("SELECT * FROM scd1_customer").fetchdf().to_string(index=False))

print("\n" + "─"*60)

# ── SCD Type 2: New row with effective dates ─────────────────────────────────
con2.execute("""
CREATE TABLE scd2_customer (
    customer_sk      INTEGER PRIMARY KEY,  -- surrogate key
    customer_id      VARCHAR(20),           -- natural/business key
    full_name        VARCHAR(200),
    city             VARCHAR(100),
    effective_date   DATE NOT NULL,
    expiry_date      DATE NOT NULL DEFAULT '9999-12-31',
    is_current       BOOLEAN NOT NULL DEFAULT TRUE
);
INSERT INTO scd2_customer VALUES
    (1, 'C001', 'Alice Wang', 'Shanghai', '2023-01-01', '2024-06-30', FALSE),
    (2, 'C001', 'Alice Wang', 'Beijing',  '2024-07-01', '9999-12-31', TRUE);
""")

print("\n=== SCD Type 2: Full history preserved ===")
print(con2.execute("SELECT * FROM scd2_customer ORDER BY effective_date").fetchdf().to_string(index=False))

# Query: what was Alice's city on 2023-12-01?
result = con2.execute("""
    SELECT city
    FROM scd2_customer
    WHERE customer_id = 'C001'
      AND '2023-12-01' BETWEEN effective_date AND expiry_date
""").fetchone()
print(f"\nAlice's city on 2023-12-01: {result[0]}")  # Shanghai
print("\n" + "─"*60)

# ── SCD Type 3: Previous value column ────────────────────────────────────────
con2.execute("""
CREATE TABLE scd3_customer (
    customer_id      VARCHAR(20) PRIMARY KEY,
    full_name        VARCHAR(200),
    current_city     VARCHAR(100),
    previous_city    VARCHAR(100),   -- only stores ONE prior value
    changed_at       TIMESTAMP
);
INSERT INTO scd3_customer VALUES ('C001', 'Alice Wang', 'Beijing', 'Shanghai', '2024-07-01');
""")
print("\n=== SCD Type 3: Current + one previous value ===")
print(con2.execute("SELECT * FROM scd3_customer").fetchdf().to_string(index=False))

In [ ]:
import pandas as pd
from datetime import date, datetime

# Implement SCD Type 2 logic in Python (simulates dbt snapshot behavior)
def apply_scd2(existing_df: pd.DataFrame, incoming_df: pd.DataFrame,
               business_key: str, tracked_cols: list[str],
               snapshot_date: date) -> pd.DataFrame:
    """
    Apply SCD Type 2 logic:
    - Rows unchanged: keep as-is
    - Rows changed: expire old row, insert new row
    - New rows: insert with effective_date = snapshot_date
    """
    INFINITY = date(9999, 12, 31)

    existing_current = existing_df[existing_df["is_current"]].copy()
    merged = existing_current.merge(
        incoming_df[[business_key] + tracked_cols],
        on=business_key, how="outer", suffixes=("_old", "_new")
    )

    records = []
    sk_counter = existing_df["sk"].max() if not existing_df.empty else 0

    for _, row in merged.iterrows():
        is_new = pd.isna(row.get("sk"))
        if is_new:
            # Brand new customer
            sk_counter += 1
            records.append({
                "sk": sk_counter,
                business_key: row[business_key],
                **{c: row[f"{c}_new"] for c in tracked_cols},
                "effective_date": snapshot_date, "expiry_date": INFINITY, "is_current": True
            })
        else:
            changed = any(row.get(f"{c}_old") != row.get(f"{c}_new") for c in tracked_cols)
            if changed:
                # Expire old row
                old = existing_current[existing_current["sk"] == row["sk"]].iloc[0].to_dict()
                old.update({"expiry_date": snapshot_date, "is_current": False})
                records.append(old)
                # Insert new row
                sk_counter += 1
                records.append({
                    "sk": sk_counter,
                    business_key: row[business_key],
                    **{c: row[f"{c}_new"] for c in tracked_cols},
                    "effective_date": snapshot_date, "expiry_date": INFINITY, "is_current": True
                })
            else:
                # Unchanged: carry forward
                records.append(
                    existing_current[existing_current["sk"] == row["sk"]].iloc[0].to_dict()
                )

    # Include expired rows that weren't in current
    expired_rows = existing_df[~existing_df["is_current"]]
    result = pd.concat([expired_rows, pd.DataFrame(records)], ignore_index=True)
    return result.sort_values(["sk"]).reset_index(drop=True)


# Initial load (Day 1)
day0_data = pd.DataFrame([
    {"sk": 1, "customer_id": "C001", "name": "Alice Wang", "city": "Shanghai",
     "effective_date": date(2024, 1, 1), "expiry_date": date(9999, 12, 31), "is_current": True},
    {"sk": 2, "customer_id": "C002", "name": "Bob Zhang",  "city": "Beijing",
     "effective_date": date(2024, 1, 1), "expiry_date": date(9999, 12, 31), "is_current": True},
])

# Incoming update: Alice moves to Shenzhen, Bob unchanged, Carol is new
incoming = pd.DataFrame([
    {"customer_id": "C001", "name": "Alice Wang", "city": "Shenzhen"},
    {"customer_id": "C002", "name": "Bob Zhang",  "city": "Beijing"},
    {"customer_id": "C003", "name": "Carol Li",   "city": "NYC"},
])

result = apply_scd2(
    day0_data, incoming,
    business_key="customer_id",
    tracked_cols=["name", "city"],
    snapshot_date=date(2024, 7, 1)
)

print("=== SCD Type 2 Result after snapshot on 2024-07-01 ===")
print(result[["sk","customer_id","name","city","effective_date","expiry_date","is_current"]].to_string(index=False))

### dbt Snapshot (Type 2) YAML 配置

dbt 的 `snapshot` 功能自动实现 SCD Type 2，配置示例：

```yaml
# snapshots/customer_snapshot.sql
{% snapshot customer_snapshot %}
  {{
    config(
      target_schema = 'snapshots',
      unique_key    = 'customer_id',
      strategy      = 'check',          -- or 'timestamp'
      check_cols    = ['city', 'email'], -- columns to track for changes
      -- strategy = 'timestamp'
      -- updated_at = 'updated_at'
    )
  }}
  SELECT customer_id, full_name, city, email
  FROM {{ source('raw', 'customers') }}
{% endsnapshot %}
```

dbt 自动维护：`dbt_scd_id`（行唯一标识）、`dbt_updated_at`、`dbt_valid_from`、`dbt_valid_to`

**面试高频追问**：`strategy='timestamp'` vs `strategy='check'` 区别？
- `timestamp`：检查 `updated_at` 字段是否变化，效率高，需要源表有可靠的更新时间戳
- `check`：逐列比较 `check_cols` 内容是否变化，不依赖时间戳，适合没有 `updated_at` 的源表

---
## 3. Data Vault 2.0

### 三大核心组件

```
   HUB_CUSTOMER ──── LINK_ORDER_CUSTOMER ──── HUB_ORDER
        │                                         │
   SAT_CUSTOMER_DETAILS               SAT_ORDER_STATUS
   SAT_CUSTOMER_CONTACT
```

| 组件 | 存储内容 | 特征 |
|------|----------|------|
| **Hub** | 业务主键（Business Key）+ 元数据 | 实体，无历史，每个业务 key 一行 |
| **Link** | Hub 之间的关系（FK 组合） | 关系，无历史，支持 M:N |
| **Satellite** | 属性 + 历史 + 源系统元数据 | 所有历史变化都记录，按属性组分拆 |

### Raw Vault vs Business Vault
- **Raw Vault**：直接从源系统加载，不做业务转换，保留原始数据
- **Business Vault**：在 Raw Vault 上做计算（派生字段、规则应用），构建业务视图

### 优缺点

| 优点 | 缺点 |
|------|------|
| 完整审计追踪（每行含 load timestamp + source） | 查询复杂（需大量 JOIN） |
| 并行加载（Hub/Link/Sat 互相独立） | 学习曲线高 |
| 灵活适应源系统变化（新属性加新 Satellite） | 不适合小数据集 |
| 支持多源整合，无破坏性变更 | 查询性能需要额外优化层 |

In [ ]:
import duckdb
import hashlib
from datetime import datetime

con3 = duckdb.connect(":memory:")

# Data Vault 2.0 DDL examples
con3.execute("""
-- HUB: Customer entity (business key + metadata, no attributes)
CREATE TABLE hub_customer (
    hub_customer_hk   VARCHAR(64) PRIMARY KEY,  -- hash of business key
    customer_id       VARCHAR(20) NOT NULL,       -- business/natural key
    load_dts          TIMESTAMP   NOT NULL,
    record_source     VARCHAR(100) NOT NULL       -- source system identifier
);

-- HUB: Order entity
CREATE TABLE hub_order (
    hub_order_hk      VARCHAR(64) PRIMARY KEY,
    order_id          VARCHAR(20) NOT NULL,
    load_dts          TIMESTAMP   NOT NULL,
    record_source     VARCHAR(100) NOT NULL
);

-- LINK: Relationship between Customer and Order (M:N supported)
CREATE TABLE link_order_customer (
    link_order_customer_hk  VARCHAR(64) PRIMARY KEY,  -- hash of both HKs
    hub_customer_hk         VARCHAR(64) NOT NULL,
    hub_order_hk            VARCHAR(64) NOT NULL,
    load_dts                TIMESTAMP   NOT NULL,
    record_source           VARCHAR(100) NOT NULL
);

-- SATELLITE: Customer contact attributes with history
CREATE TABLE sat_customer_contact (
    hub_customer_hk   VARCHAR(64) NOT NULL,
    load_dts          TIMESTAMP   NOT NULL,
    load_end_dts      TIMESTAMP,              -- NULL = current record
    hash_diff         VARCHAR(64) NOT NULL,   -- hash of all tracked attributes
    record_source     VARCHAR(100) NOT NULL,
    -- Attributes
    email             VARCHAR(200),
    phone             VARCHAR(50),
    city              VARCHAR(100),
    PRIMARY KEY (hub_customer_hk, load_dts)
);
""")

# Helper: compute DV hash key
def dv_hash(*values: str) -> str:
    combined = "|".join(str(v).upper().strip() for v in values)
    return hashlib.sha256(combined.encode()).hexdigest()

now = datetime(2024, 7, 1, 8, 0, 0)
source = "CRM_SYSTEM"

# Load Hub: Customer
customers = [("C001", "alice@email.com", "13900139000", "Shanghai"),
             ("C002", "bob@email.com",   "13800138000", "Beijing")]

for cid, email, phone, city in customers:
    hk = dv_hash(cid)
    con3.execute(f"""
        INSERT OR IGNORE INTO hub_customer VALUES
        ('{hk}', '{cid}', '{now}', '{source}');
    """)
    diff = dv_hash(email, phone, city)
    con3.execute(f"""
        INSERT INTO sat_customer_contact VALUES
        ('{hk}', '{now}', NULL, '{diff}', '{source}', '{email}', '{phone}', '{city}');
    """)

print("=== Data Vault: HUB_CUSTOMER ===")
print(con3.execute("SELECT hub_customer_hk[:16] || '...', customer_id, load_dts, record_source FROM hub_customer").fetchdf().to_string(index=False))

print("\n=== Data Vault: SAT_CUSTOMER_CONTACT (current records) ===")
print(con3.execute("""
    SELECT h.customer_id, s.email, s.phone, s.city, s.load_dts
    FROM hub_customer h
    JOIN sat_customer_contact s ON h.hub_customer_hk = s.hub_customer_hk
    WHERE s.load_end_dts IS NULL
""").fetchdf().to_string(index=False))

---
## 4. OBT (One Big Table) vs 规范化

### OBT：一张超宽表

将所有维度属性和指标 **反规范化** 到单张宽表中：

```sql
-- OBT: 所有信息在一行
SELECT
    order_id, order_date, order_status,
    customer_id, customer_name, customer_city, customer_segment,
    product_id, product_name, category, brand,
    quantity, unit_price, net_amount
FROM mart_orders_obt
WHERE order_date >= '2024-01-01';
```

### 对比

| 维度 | OBT | 规范化（星型/雪花） |
|------|-----|-------------------|
| 查询简单度 | 极简，无需 JOIN | 需要多表 JOIN |
| BI 工具友好 | 极好（单表拖拽） | 需要预定义关系 |
| 列数 | 爆炸（可能 200+ 列） | 受控 |
| 存储冗余 | 高（维度值重复） | 低 |
| 维护成本 | 高（改一个维度需重新构建宽表） | 低（改维度表即可） |
| 列式存储效率 | 好（Parquet/Delta 压缩率高） | 中 |
| 适用场景 | 小中型数据集、单一用例、探索分析 | 大型 DWH、多用例、复杂业务 |

### 何时使用 OBT？
- 数据集规模 < 数十 GB
- 用途单一（如某部门的专项分析报表）
- 数据新鲜度要求不高（T+1 批量重建可接受）
- 团队 SQL 能力有限，希望最简单的访问方式

**面试要点**：OBT 不是反模式，是一种有意识的工程权衡。在 Lakehouse + 列式存储的场景下，OBT 的存储惩罚被大幅压缩，而查询简单性的收益很明显。

In [ ]:
import duckdb
import pandas as pd

con4 = duckdb.connect(":memory:")

# Simulate building an OBT from star schema
# (Reusing the star schema tables from the first cell — recreate here for standalone)
con4.execute("""
CREATE TABLE dim_date_t    AS SELECT 20240101 AS date_key, DATE '2024-01-01' AS full_date, 2024 AS year, 1 AS month, 'January' AS month_name;
CREATE TABLE dim_product_t AS SELECT 1 AS product_key, 'P001' AS product_id, 'Laptop Pro 15' AS product_name, 'Electronics' AS category, 'TechBrand' AS brand;
CREATE TABLE dim_customer_t AS SELECT 1 AS customer_key, 'C001' AS customer_id, 'Alice Wang' AS full_name, 'Shanghai' AS city, 'China' AS country, 'RETAIL' AS segment;
CREATE TABLE fact_sales_t  AS SELECT 1 AS sale_id, 20240101 AS date_key, 1 AS product_key, 1 AS customer_key, 2 AS quantity, 1200.00 AS unit_price, 2400.00 AS gross_amount, 100.00 AS discount_amt, 2300.00 AS net_amount;

-- Build OBT: denormalize everything into one wide table
CREATE TABLE mart_orders_obt AS
SELECT
    f.sale_id,
    -- Date attributes
    d.full_date     AS order_date,
    d.year          AS order_year,
    d.month_name    AS order_month,
    -- Product attributes
    p.product_id,
    p.product_name,
    p.category      AS product_category,
    p.brand         AS product_brand,
    -- Customer attributes
    c.customer_id,
    c.full_name     AS customer_name,
    c.city          AS customer_city,
    c.country       AS customer_country,
    c.segment       AS customer_segment,
    -- Measures
    f.quantity,
    f.unit_price,
    f.gross_amount,
    f.discount_amt,
    f.net_amount
FROM fact_sales_t   f
JOIN dim_date_t     d ON f.date_key     = d.date_key
JOIN dim_product_t  p ON f.product_key  = p.product_key
JOIN dim_customer_t c ON f.customer_key = c.customer_key;
""")

obt = con4.execute("SELECT * FROM mart_orders_obt").fetchdf()
print("=== OBT: One Big Table (all attributes in one row) ===")
print(f"Columns: {list(obt.columns)}")
print(f"\nTotal columns: {len(obt.columns)}")
print("\nSample row:")
for col, val in obt.iloc[0].items():
    print(f"  {col:<25} = {val}")

# OBT query: no JOIN needed
print("\n=== OBT Query (NO JOIN required): Sales by Category ===")
result = con4.execute("""
    SELECT product_category, SUM(net_amount) AS total_sales
    FROM mart_orders_obt
    GROUP BY product_category
""").fetchdf()
print(result.to_string(index=False))

---
## 5. Lakehouse 架构设计

### 核心思想

**Data Lake**（便宜存储 + 原始数据）+ **Data Warehouse**（ACID + Schema + BI）的优势结合：

```
数据源
  │  Kafka / CDC / API / Files
  ▼
┌─────────────────────────────────────────────────────────────┐
│  BRONZE Layer (Raw)                                         │
│  · 原始数据，不做转换                                        │
│  · 格式：Parquet / Delta / Iceberg                          │
│  · 保留所有历史，分区按 ingestion_date                       │
└──────────────────────┬──────────────────────────────────────┘
                       │  清洗 / 去重 / 类型转换
                       ▼
┌─────────────────────────────────────────────────────────────┐
│  SILVER Layer (Cleaned & Conformed)                         │
│  · 数据质量验证，JOIN 整合多源                               │
│  · Schema 强制，NULL 处理，标准化                            │
│  · 支持 Time Travel（Delta/Iceberg）                        │
└──────────────────────┬──────────────────────────────────────┘
                       │  聚合 / 业务逻辑 / 指标计算
                       ▼
┌─────────────────────────────────────────────────────────────┐
│  GOLD Layer (Business Mart)                                 │
│  · 面向业务的聚合表 / 星型 schema / OBT                     │
│  · Tableau / PowerBI / Looker 直连                          │
│  · SLA 要求最高（数据新鲜度、准确性）                        │
└─────────────────────────────────────────────────────────────┘

计算引擎：Apache Spark / Trino / Flink
表格式：  Delta Lake / Apache Iceberg / Apache Hudi
编排：    dbt (SQL transforms) + Airflow / Prefect (orchestration)
存储：    S3 / GCS / ADLS
```

### 关键技术选型

| 组件 | 选项 | 核心能力 |
|------|------|----------|
| 表格式 | Delta Lake / Iceberg | ACID 事务、Time Travel、Schema Evolution |
| 批处理 | Spark / dbt | 大规模 ETL、SQL 转换 |
| 流处理 | Flink / Spark Streaming | 实时摄入到 Bronze/Silver |
| 即席查询 | Trino / Athena | 快速 SQL on S3 |
| 数据目录 | Glue Catalog / Hive Metastore | Schema 注册、表发现 |
| 编排 | Airflow / Prefect | DAG 调度、依赖管理 |

### Delta Lake vs Iceberg 选型

| 维度 | Delta Lake | Apache Iceberg |
|------|------------|----------------|
| 生态 | Databricks 优先 | 厂商中立 |
| 查询引擎 | Spark 最优 | Spark/Trino/Flink/Hive |
| Merge 性能 | 优秀 | 良好 |
| Time Travel | `VERSION AS OF` | `AS OF TIMESTAMP` |
| Hidden Partitioning | 不支持 | 支持（无需用户管理分区列） |
| 推荐场景 | Databricks 环境 | 多引擎、云中立环境 |

In [ ]:
import duckdb
import pandas as pd
from datetime import datetime, date

# Simulate Bronze → Silver → Gold pipeline with DuckDB
# (In production: replace DuckDB with Spark + Delta Lake)

con5 = duckdb.connect(":memory:")

# ── BRONZE: Raw ingestion (no transformation, preserve as-is) ─────────────────
bronze_raw = pd.DataFrame([
    {"ingestion_ts": "2024-01-01T08:00:00", "event_type": "order_placed",
     "order_id": "ORD001", "customer_id": "C001", "amount": "1200.00", "currency": "CNY"},
    {"ingestion_ts": "2024-01-01T08:05:00", "event_type": "order_placed",
     "order_id": "ORD002", "customer_id": "C002", "amount": "350.50",  "currency": "USD"},
    {"ingestion_ts": "2024-01-01T08:10:00", "event_type": "order_placed",
     "order_id": "ORD001", "customer_id": "C001", "amount": "1200.00", "currency": "CNY"},  # duplicate
    {"ingestion_ts": "2024-01-01T08:15:00", "event_type": "order_cancelled",
     "order_id": None,     "customer_id": "C003", "amount": None,       "currency": "CNY"},  # bad data
])
con5.register("bronze_events_raw", bronze_raw)
print("=== BRONZE Layer (raw, unvalidated) ===")
print(bronze_raw.to_string(index=False))

# ── SILVER: Clean, deduplicate, type-cast ────────────────────────────────────
silver = con5.execute("""
    SELECT
        CAST(ingestion_ts AS TIMESTAMP) AS event_ts,
        event_type,
        order_id,
        customer_id,
        CAST(amount AS DECIMAL(12,2))   AS amount_local,
        currency,
        -- Normalize to USD (simplified FX)
        CASE currency
            WHEN 'CNY' THEN CAST(amount AS DECIMAL(12,2)) / 7.1
            ELSE             CAST(amount AS DECIMAL(12,2))
        END AS amount_usd,
        CURRENT_TIMESTAMP AS processed_ts
    FROM bronze_events_raw
    WHERE order_id IS NOT NULL         -- reject bad rows
      AND amount   IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY order_id, event_type
        ORDER BY ingestion_ts
    ) = 1                              -- deduplicate
""").fetchdf()

print("\n=== SILVER Layer (cleaned, deduplicated, typed) ===")
print(silver.to_string(index=False))

con5.register("silver_events", silver)

# ── GOLD: Aggregated business mart ───────────────────────────────────────────
gold = con5.execute("""
    SELECT
        customer_id,
        COUNT(*)                 AS total_orders,
        SUM(amount_usd)          AS total_gmv_usd,
        AVG(amount_usd)          AS avg_order_value_usd,
        MIN(event_ts)::DATE      AS first_order_date
    FROM silver_events
    WHERE event_type = 'order_placed'
    GROUP BY customer_id
    ORDER BY total_gmv_usd DESC
""").fetchdf()

print("\n=== GOLD Layer (business mart: customer GMV summary) ===")
print(gold.to_string(index=False))

print("\n=== Layer Summary ===")
print(f"Bronze rows: {len(bronze_raw)} (raw, includes duplicates & bad data)")
print(f"Silver rows: {len(silver)} (after dedup + quality filter)")
print(f"Gold rows:   {len(gold)} (aggregated, ready for BI)")

---
## 复习要点

### Kimball 维度建模
- **星型**：维度反规范化，查询少 JOIN，BI 最友好，存储略大
- **雪花型**：维度规范化，存储小，JOIN 多，适合维度层级复杂场景
- **Galaxy**：多事实表共享一致性维度，企业级 DWH 标准模式
- 代理键（Surrogate Key）vs 业务键（Natural Key）：代理键隔离源系统变化

### SCD Types
- **Type 1**：覆写，无历史，适合纠错
- **Type 2**：新行 + 有效期，最常用，dbt snapshot 实现
- **Type 3**：加列保存前值，只保一次历史
- **Type 4**：主表 + 历史表，访问模式分离
- **Type 6**：1+2+3 组合，最灵活最复杂
- 面试："你们用 Type 2 时如何处理晚到数据？" → 需要重跑 snapshot + 级联更新

### Data Vault 2.0
- Hub（业务主键）+ Link（关系）+ Satellite（属性+历史）
- 核心优势：并行加载、完整审计、无破坏性变更
- 适合大型企业、多源整合、合规要求高的场景
- 查询复杂是主要缺点，通常在 Business Vault 层做视图优化

### OBT vs 规范化
- OBT 不是反模式，是有意识的权衡
- 列式存储（Parquet/Delta）让 OBT 冗余的存储惩罚大幅降低
- 小数据集、单用例、BI 团队优先 → OBT
- 大型 DWH、多用例、复杂业务 → 规范化 + 星型

### Lakehouse
- Bronze（原始）→ Silver（清洗）→ Gold（业务 Mart）
- 关键能力：ACID 事务、Time Travel、Schema Evolution（Delta/Iceberg）
- Delta Lake = Databricks 生态最优；Iceberg = 多引擎云中立
- dbt 做 SQL 转换，Spark/Trino 做计算，Airflow 做编排

---
## 练习

以下练习覆盖本章所有主题，建议先独立作答再参考提示。

### 练习 1 — Schema 设计选型

你负责为一家电商公司设计数仓，业务场景如下：
- 核心业务：订单分析、用户行为分析、供应链库存分析
- 数据量：每天 500 万订单，维度表最大 2000 万行
- 用户：BI 团队（50 人，主要用 Tableau）、数据科学团队（10 人，写 SQL）
- 要求：订单分析需要按地区/品类/时间多维下钻，库存分析需要按仓库/SKU 追踪

**问题**：
1. 选择星型、雪花型还是 Galaxy 模式？为什么？
2. 设计订单事实表和至少 3 张维度表的 DDL（写 SQL）
3. 这个设计中 `order_id` 应该作为事实表的主键吗？为什么？

**评分标准**：
- 选型有明确理由（不只是说好或坏）
- DDL 包含代理键、业务键、正确的数据类型
- 理解为什么事实表通常不以 `order_id` 为主键（可能一个订单多行：如退款行）

In [ ]:
# Exercise 1: Design your star/galaxy schema DDL here
import duckdb
con = duckdb.connect(":memory:")

# TODO: Write DDL for fact and dimension tables
# Example structure (fill in the details):

# con.execute("""
# CREATE TABLE dim_... (
#     ...  -- surrogate key
#     ...  -- business key
#     ...  -- attributes
# );
# CREATE TABLE fact_orders (
#     ...  -- grain: one row per order LINE ITEM, not per order
# );
# """)

print("Exercise 1: Implement your schema design above")

### 练习 2 — SCD Type 2 MERGE 语句

给定以下 `dim_employee` 表（SCD Type 2），写一个 MERGE 语句：
- 如果员工已存在且部门/级别发生变化：将旧记录的 `expiry_date` 更新为今天，插入新记录
- 如果员工已存在但无变化：不操作
- 如果是新员工：直接插入

```sql
-- dim_employee (SCD Type 2)
-- employee_id (natural key), name, department, level,
-- effective_date, expiry_date, is_current
```

**追加问题**：如果源系统没有 `updated_at` 字段，dbt snapshot 应该用哪种 strategy？有什么代价？

In [ ]:
import duckdb
con = duckdb.connect(":memory:")

# Setup: existing dim_employee table
con.execute("""
CREATE TABLE dim_employee (
    employee_sk    INTEGER PRIMARY KEY,
    employee_id    VARCHAR(20) NOT NULL,
    name           VARCHAR(200),
    department     VARCHAR(100),
    level          VARCHAR(50),
    effective_date DATE,
    expiry_date    DATE DEFAULT '9999-12-31',
    is_current     BOOLEAN DEFAULT TRUE
);
INSERT INTO dim_employee VALUES
    (1, 'E001', 'Alice', 'Engineering', 'L5', '2023-01-01', '9999-12-31', TRUE),
    (2, 'E002', 'Bob',   'Marketing',   'L4', '2023-01-01', '9999-12-31', TRUE);

-- Source: incoming data (Alice promoted to L6, Carol is new)
CREATE TABLE src_employee AS
SELECT * FROM (VALUES
    ('E001', 'Alice', 'Engineering', 'L6'),
    ('E002', 'Bob',   'Marketing',   'L4'),
    ('E003', 'Carol', 'Data',        'L3')
) t(employee_id, name, department, level);
""")

# TODO: Implement the SCD Type 2 MERGE logic
# Hint: DuckDB does not have native MERGE — use INSERT + UPDATE pattern
# Step 1: Identify changed records
# Step 2: Expire old records
# Step 3: Insert new versions
# Step 4: Insert brand new records

print("Before:")
print(con.execute("SELECT * FROM dim_employee ORDER BY employee_id, effective_date").fetchdf().to_string(index=False))

# Your SCD logic here

print("\nAfter (expected: Alice has 2 rows, Bob unchanged, Carol added):")
# print(con.execute("SELECT * FROM dim_employee ORDER BY employee_id, effective_date").fetchdf().to_string(index=False))

### 练习 3 — Data Vault 设计

你在整合来自 3 个系统的数据：CRM（客户）、ERP（订单）、WMS（仓储）。
同一个客户在三个系统中 ID 不同：`CRM_ID`、`ERP_CUST_NO`、`WMS_CLIENT_CODE`。

**问题**：
1. 为 Data Vault 设计 Hub/Link/Satellite 结构（画出实体关系图或写 DDL）
2. 如何处理同一客户在三个系统中的 ID 映射？（提示：Same-As Link 或 Identity Hub）
3. 如果 CRM 系统新增了一个 `loyalty_tier` 字段，你如何在不破坏现有加载流程的情况下添加？

**评分标准**：
- 正确区分 Hub/Link/Satellite 的职责
- 理解 Data Vault 如何处理多源同一实体
- 理解 Satellite 分拆的原则（按更新频率 / 按源系统 / 按属性组）

In [ ]:
# Exercise 3: Design Data Vault DDL for multi-source integration
import duckdb
con = duckdb.connect(":memory:")

# TODO: Design Hub, Link, Satellite tables for CRM + ERP + WMS integration
# Consider:
# - hub_customer (which business key to use? one hub per source or merged?)
# - How to map CRM_ID <-> ERP_CUST_NO <-> WMS_CLIENT_CODE
# - Satellites per source system vs merged satellite

# Example starter:
# con.execute("""
# -- Hub: one per source system OR one unified hub?
# -- (Common pattern: one hub with business key from master system,
#  --  plus a Reference table for cross-system ID mapping)
# CREATE TABLE hub_customer (
#     hub_customer_hk  VARCHAR(64) PRIMARY KEY,
#     ...
# );
# """)

print("Exercise 3: Design your Data Vault structure here")

### 练习 4 — OBT vs 星型选型辩论

面试官问：
> "我们的 Gold Layer 现在有 30 张星型维度表和 5 张事实表，BI 用户抱怨说 JOIN 太多，查询太慢。
> 有人建议全部合并成 OBT。你怎么看？"

**任务**：用 150-250 字写一个结构化回答，包含：
1. OBT 的优缺点（结合这个具体场景）
2. 你会问哪些澄清性问题才能做出最终建议？
3. 如果不用 OBT，还有什么其他方案解决慢查询问题？

**评分标准**：
- 不直接给答案，而是先澄清场景（数据量、查询模式、更新频率）
- 提出 OBT + 保留星型的混合策略
- 提到其他方案：物化视图、预聚合、分区优化、列裁剪

**你的回答（在此处写作）**：

```
首先，我会问几个澄清性问题：...

对于 OBT 的优缺点：...

其他方案：...

我的建议：...
```

### 练习 5 — Lakehouse 架构问答

**场景**：你的 Lakehouse 使用 Delta Lake + Spark + dbt + Airflow。Bronze Layer 每小时从 Kafka 摄入，Silver/Gold 每天 T+1 构建。

**问题（逐一回答）**：

1. 用户反映某个 Gold 表的数据在今天 06:00 之前是正确的，之后变错了。你如何快速回滚到 06:00 的版本？写出 Delta Lake 或 Iceberg 的 SQL。

2. 你的 Bronze Layer Parquet 文件越来越多（每小时一批，每批 10,000 个小文件），查询性能下降。你如何解决这个 Small File Problem？

3. Silver Layer 需要对一个大表增加一列（`country_code VARCHAR(5)`），现有的 Spark 作业和 dbt 模型不应该中断。如何实现 Schema Evolution？

4. 你如何在 Lakehouse 中实现**数据血缘（Data Lineage）追踪**？列出至少 3 种方法（工具或手动实现）。

In [ ]:
# Exercise 5: Demonstrate Delta Lake / Iceberg concepts with DuckDB simulation

import duckdb
import pandas as pd
from datetime import datetime

con = duckdb.connect(":memory:")

# Q1: Simulate Time Travel (Delta Lake: RESTORE TABLE ... VERSION AS OF)
# In real Delta Lake:
#   RESTORE TABLE gold.orders TO VERSION AS OF 5;
#   RESTORE TABLE gold.orders TO TIMESTAMP AS OF '2024-01-01 06:00:00';
#   SELECT * FROM gold.orders TIMESTAMP AS OF '2024-01-01 06:00:00';

# Simulate with versioned snapshots in DuckDB
print("=== Q1: Time Travel simulation ===")
con.execute("""
CREATE TABLE gold_orders_v1 AS SELECT 'ORD001' AS order_id, 1200.0 AS amount, 'correct' AS status;
CREATE TABLE gold_orders_v2 AS SELECT 'ORD001' AS order_id, 9999.0 AS amount, 'corrupted' AS status;
""")
print("v2 (corrupted):", con.execute("SELECT * FROM gold_orders_v2").fetchdf().to_string(index=False))
print("Restore to v1:",  con.execute("SELECT * FROM gold_orders_v1").fetchdf().to_string(index=False))
print()
print("Delta Lake SQL (production):")
print("  RESTORE TABLE gold.orders TO TIMESTAMP AS OF '2024-01-01 06:00:00';")
print("  -- or: SELECT * FROM gold.orders VERSION AS OF 5")

# Q2: Small File Problem — demonstrate OPTIMIZE / VACUUM
print("\n=== Q2: Small File Problem Solutions ===")
solutions = [
    ("Delta OPTIMIZE",     "OPTIMIZE delta.`s3://bucket/bronze/events` ZORDER BY (customer_id, event_date);"),
    ("Iceberg Rewrite",    "CALL system.rewrite_data_files('db.bronze_events');"),
    ("Spark coalesce",     "df.coalesce(10).write.mode('overwrite').parquet('s3://...')"),
    ("Auto-compact",       "spark.databricks.delta.autoCompact.enabled = true"),
]
for name, sql in solutions:
    print(f"  [{name}]: {sql}")

# Q3: Schema Evolution
print("\n=== Q3: Schema Evolution ===")
print("Delta Lake: spark.write.option('mergeSchema', 'true').format('delta').save(...)")
print("ALTER TABLE silver.customers ADD COLUMN country_code VARCHAR(5);  -- Iceberg/Delta")
print("dbt: add column to model SELECT with COALESCE(country_code, 'UNK') for backward compat")

print("\nExercise 5: Review the Delta Lake / Iceberg concepts above")